In [ ]:
# Confirm the analysis scope, array orientation, and motion-censoring pattern.
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

SEED = 28
ROOT = Path.cwd()
RAW = ROOT / "data" / "raw"
assert RAW.exists(), "Start Jupyter from the repository root."

In [ ]:
participants = pd.read_csv(RAW / "participants.tsv", sep="\t", na_values=["n/a"])
participants.groupby("HormoneStatus")["session_id"].count()

In [ ]:
def session_number(path):
    return int(re.search(r"ses-(\d+)", path.name).group(1))

files = sorted(RAW.glob("*.npy"), key=session_number)
rows = []
for path in files:
    x = np.load(path)
    missing = np.isnan(x)
    rows.append({
        "session": session_number(path),
        "n_regions": x.shape[0],
        "n_trs": x.shape[1],
        "missing_trs": missing.any(axis=0).sum(),
        "partial_missing_trs": (missing.any(axis=0) & ~missing.all(axis=0)).sum(),
    })

inventory = pd.DataFrame(rows)
inventory.head()

In [ ]:
natural = inventory.query("session <= 30")
display(natural.describe())
assert len(files) == 60
assert (inventory[["n_regions", "n_trs"]] == [100, 820]).all().all()
assert inventory["partial_missing_trs"].eq(0).all()

In [ ]:
hormones = participants.iloc[:30].copy()
hormones["session"] = hormones["session_id"].str.removeprefix("ses-").astype(int)
fig, axes = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
sns.lineplot(hormones, x="session", y="Estradiol (LCMS; pg/mL)", marker="o", ax=axes[0])
sns.lineplot(hormones, x="session", y="Progesterone (LCMS; ng/mL)", marker="o", ax=axes[1])
fig.tight_layout()